# Finding similar items
The aim is to detect similar textual items in the `text` field of the Kaggle [Yelp](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset) dataset. To be able to download the dataset we ask you to insert below your Kaggle username and password.

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = "luciaannamellini"
os.environ['KAGGLE_KEY'] = "aa8ef96186c55a415b84467ee6e86a4a"

In [ ]:
import string
import zipfile

foldername = "/content/yelp-dataset"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init("spark-3.5.0-bin-hadoop3")# SPARK_HOME
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We first of all import the Yelp dataset from Kaggle

In [ ]:

!kaggle datasets download -d yelp-dataset/yelp-dataset

100% 4.06G/4.07G [00:44<00:00, 121MB/s]
100% 4.07G/4.07G [00:45<00:00, 97.0MB/s]


In [ ]:
with zipfile.ZipFile(foldername + ".zip", 'r') as zip_ref:
    zip_ref.extractall(foldername )
os.remove(foldername + ".zip")

We take into consideration the portion of the dataset regarding reviews, contained in `yelp_academic_dataset_review.json`, and we keep the resulting dataframe in RDD form.

In [ ]:
df_reviews = spark.read.json(foldername + "/yelp_academic_dataset_review.json")
rdd_reviews = df_reviews.rdd

Seen the aim of the project we will only be looking at the `text` attribute of the imported reviews.

In [ ]:
rdd_reviews_text = rdd_reviews.map(lambda x: x['text'])

## Data pre-processing

We begin by removing text cells that are `None` or that contain empty strings. It is possible to verify that for each review the text is non-empty.

In [ ]:
rdd_reviews_text = rdd_reviews_text.filter(lambda text: bool(text))

To arrive at a list of significant words for the text of each review we apply the following modifications:
* remove all punctuation and control characters,
* set all characters to lowercase,
* obtain all words from the text paragraph,
* remove all stop words,
* consider each word once.

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

#list of transformations to apply to the set of review texts
remove_punctuation = lambda text: text.translate(str.maketrans('', '', string.punctuation+"\n"))
text_to_lowercase = lambda text: text.lower()
text_to_words = lambda text: text.split(" ")
remove_stopwords = lambda wordlist: [word for word in wordlist if word not in stopwords.words('english')]
remove_duplicates = lambda wordlist: list(set(wordlist))
preprocessing = [remove_punctuation, text_to_lowercase, text_to_words, remove_stopwords, remove_duplicates]

for func in preprocessing:
    rdd_reviews_text = rdd_reviews_text.map(func)